# FABN SAP Optimizer — Phase 2: Daily Dynamic Re-Optimization Backtest

Tests the thesis of `SAP.pdf`: does **daily** dynamic re-optimization beat a static buy-and-hold
portfolio under statutory (book-yield) economics? Starting from cash on day 1 we build an optimal
book, then **every trading day** we re-optimize and take any opportunity that — net of **trading
cost, capital-cost change, borrowing cost, and lending revenue** — improves value. Realized net
statutory income is accrued daily and accumulated.

---

## 1. The trade decision (what changed vs the quarterly version)

Each day we hold $h^{prev}$ and choose **buys** $b_i\ge0$ and **sells** $0\le s_i\le h^{prev}_i$, so
$$h_i = h^{prev}_i + b_i - s_i .$$

**Retained vs new lots (amortized cost).** Under SAP a bond's book yield is locked at purchase. So:
- retained notional $(h^{prev}_i-s_i)$ keeps its **locked** in-force yield $y^{bk}_i$;
- newly bought notional $b_i$ enters at today's **market** yield $y^{mkt}_i$.

The decision therefore compares *what a new bond yields* against *the locked yield you give up* — the
true pickup — not market-vs-market. This is linear in $(b_i,s_i)$.

**Decision objective** (horizon $T$ = years remaining to FABN maturity; a holding is valued over the
life it would actually earn over, so a one-time trade cost is weighed against horizon income):
$$
\max_{b,s\ge0}\;
T\!\sum_i\!\big[(h^{prev}_i-s_i)(y^{bk}_i-r^F)+b_i(y^{mkt}_i-r^F)\big]
-T\lambda\!\sum_i\theta_i h_i
-\sum_i\tau_i(b_i+s_i)
+\,r_{save}\,\delta\!\sum_q B_q
-\,r_{borrow}\,\delta\!\sum_q s^{net}_q
$$

So all four of your terms enter the swap decision: **trading cost** $\tau_i(b_i+s_i)$,
**capital-cost change** $T\lambda\theta_i h_i$, **lending revenue** $r_{save}\delta\sum B_q$,
**borrowing cost** $r_{borrow}\delta\sum s^{net}_q$.

Constraints (linear): budget $\sum h_i=H$; sell cap $s_i\le h^{prev}_i$; duration band
$\lvert\sum D_i h_i - D^L H\rvert\le\varepsilon_D H$; issuer cap $\sum_{i\in g}h_i\le\delta_{iss}H$;
lending-facility balance recursion to maturity; PV-shortfall cap $\sum_q DF_q s^{net}_q\le\phi\,PV(L)$.

---

## 2. Realized daily P&L (accrual — kept separate, no double counting)

Over the day $[t_k,t_{k+1}]$ of length $\Delta_k$ (years), after the trade and the book-yield ledger
update:
$$
\text{Net}_k=\Delta_k\!\sum_i h_{k,i}\big(y^{bk}_{k,i}-r^F\big)
-\Delta_k\,\lambda\!\sum_i\theta_i h_{k,i}
-\sum_i\tau_{k,i}(b_{k,i}+s_{k,i})
$$
The book-yield ledger blends retained and bought lots:
$y^{bk}_{k,i}=\big[(h^{prev}_i-s_i)y^{bk}_{k-1,i}+b_i\,y^{mkt}_{k,i}\big]/h_{k,i}$.

Income accrues at the **locked** yield; the facility interest is a *decision-shaping* / ALM term
(full-horizon), so it is **not** re-accrued daily. **Matured** bonds redeem at par (no spread); their
cash is redeployed (dynamic) or held as cash earning $r^F$, i.e. net-zero spread (static).

**Honest limitations (unchanged):** IMR/AVR excluded — sale gains/losses not recognized, so this is a
*conservative* test (only forward book-yield pickup net of costs, not trading gains). Prices treated
as clean. The dynamic-vs-static outcome is data-driven, not assumed.

---
## Section 0 — Imports & Bulk Data Load (BigQuery + FRED, once)

In [ ]:
import numpy as np, pandas as pd, time, warnings
import gurobipy as gp
from gurobipy import GRB
from google.cloud import bigquery
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import pandas_datareader.data as web
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

PROJECT_ID    = "insurance-backed-securities"
FABN_ISSUE    = pd.Timestamp("2022-09-06")
FABN_MATURITY = pd.Timestamp("2027-09-06")
FABN_COUPON   = 0.03205
r_FABN        = FABN_COUPON
H             = 500_000_000.0
client = bigquery.Client(project=PROJECT_ID)
print("Connected:", client.project)

In [ ]:
# Static attributes
fixed = client.query(f"""
    SELECT CUSIP, `Amt Out` AS amt_out, Cpn AS coupon, Maturity AS maturity,
           `BBG Composite` AS rating_sp, `Mac Dur _Ask_` AS mac_dur_bbg,
           RTG_MOODY AS rating_moodys, BICS_LEVEL_1_SECTOR_NAME AS sector
    FROM `{PROJECT_ID}.Securities.Agg_Fixed_Field` WHERE CUSIP IS NOT NULL
""").to_dataframe().drop_duplicates(subset="CUSIP").reset_index(drop=True)
fixed["maturity"] = pd.to_datetime(fixed["maturity"])
CUSIPS = fixed["CUSIP"].tolist(); N = len(CUSIPS); cidx = {c:i for i,c in enumerate(CUSIPS)}
maturity64 = fixed.set_index("CUSIP").loc[CUSIPS,"maturity"].values.astype("datetime64[ns]")
annual_cpn = fixed.set_index("CUSIP").loc[CUSIPS,"coupon"].values/100.0
print(f"Universe N = {N}")

In [ ]:
# C-1 RBC factor (theta)
C1_SP={"AAA":0.00158,"AA+":0.00271,"AA":0.00419,"AA-":0.00523,"A+":0.00657,"A":0.00816,"A-":0.01016,
       "BBB+":0.01261,"BBB":0.01523,"BBB-":0.02168,"BB+":0.03151,"BB":0.04537,"BB-":0.06017,"B+":0.07386,
       "B":0.09535,"B-":0.12428,"CCC+":0.16942,"CCC":0.23798,"CCC-":0.32975,"D":0.30000}
C1_MO={"Aaa":0.00158,"Aa1":0.00271,"Aa2":0.00419,"Aa3":0.00523,"A1":0.00657,"A2":0.00816,"A3":0.01016,
       "Baa1":0.01261,"Baa2":0.01523,"Baa3":0.02168,"Ba1":0.03151,"Ba2":0.04537,"Ba3":0.06017,"B1":0.07386,
       "B2":0.09535,"B3":0.12428,"Caa1":0.16942,"Caa2":0.23798,"Caa3":0.32975,"Ca":0.30000,"C":0.30000}
def _c1(sp,mo):
    if pd.notna(sp) and str(sp).strip() in C1_SP: return C1_SP[str(sp).strip()]
    if pd.notna(mo) and str(mo).strip() in C1_MO: return C1_MO[str(mo).strip()]
    return C1_SP["BBB"]
theta = np.array([_c1(fixed.loc[fixed.CUSIP==c,"rating_sp"].values[0],
                      fixed.loc[fixed.CUSIP==c,"rating_moodys"].values[0]) for c in CUSIPS])
ISSUER_GROUPS = {}
for i,c in enumerate(CUSIPS): ISSUER_GROUPS.setdefault(c[:6],[]).append(i)
print(f"theta range: {theta.min():.5f} – {theta.max():.5f}  | {len(ISSUER_GROUPS)} issuers")

In [ ]:
# Asset cash flows (per 100 face), pre-grouped per CUSIP for fast IRR
cf_all = client.query(f"""
    SELECT PaymentDate, CUSIP, Payment FROM `{PROJECT_ID}.Securities.Asset_Cashflows`
""").to_dataframe()
cf_all["PaymentDate"] = pd.to_datetime(cf_all["PaymentDate"])
cf_all = cf_all.groupby(["PaymentDate","CUSIP"], as_index=False)["Payment"].sum()
CF_BY = {}
for c,g in cf_all.groupby("CUSIP"):
    g = g.sort_values("PaymentDate")
    CF_BY[c] = (g["PaymentDate"].values.astype("datetime64[ns]"), g["Payment"].values/100.0)  # per $1 face
print(f"Cash-flow rows: {len(cf_all):,}")

In [ ]:
# Daily price panels, wide (index=Date, cols=CUSIP), aligned to CUSIPS
def _wide(table, cast=False):
    expr = "SAFE_CAST(Price AS FLOAT64)" if cast else "Price"
    df = client.query(f"SELECT Date,CUSIP,{expr} AS Price FROM `{table}`").to_dataframe()
    df["Date"] = pd.to_datetime(df["Date"])
    return df.pivot_table(index="Date",columns="CUSIP",values="Price").sort_index().reindex(columns=CUSIPS)
mid_w = _wide(f"{PROJECT_ID}.Mid_Price.mid_long_raw")
bid_w = _wide(f"{PROJECT_ID}.Bid_Price.bid_long_raw")
ask_w = _wide(f"{PROJECT_ID}.Ask_Price.ask_long_raw", cast=True)
PANEL_DATES = mid_w.index
print(f"Price panel: {mid_w.shape[0]} days x {mid_w.shape[1]} bonds "
      f"({PANEL_DATES.min().date()} → {PANEL_DATES.max().date()})")

In [ ]:
# FRED Treasury curve history (one call; retry + static fallback)
MAT_YRS=[1/12,3/12,6/12,1,2,3,5,7,10,20,30]
FRED_T=["DGS1MO","DGS3MO","DGS6MO","DGS1","DGS2","DGS3","DGS5","DGS7","DGS10","DGS20","DGS30"]
FRED_FALLBACK=[4.40,4.35,4.26,4.19,4.27,4.34,4.45,4.55,4.66,4.95,4.88]
def _fred(n=3):
    for k in range(1,n+1):
        try:
            raw=web.DataReader(FRED_T,"fred",start=PANEL_DATES.min()-pd.Timedelta(days=10),
                               end=PANEL_DATES.max()+pd.Timedelta(days=10))
            raw.columns=MAT_YRS; raw=raw.dropna(how="all")
            if raw.empty: raise ValueError("empty")
            print(f"FRED loaded (attempt {k}): {raw.shape[0]} days"); return raw
        except Exception as e:
            print(f"  FRED attempt {k}/{n} failed: {type(e).__name__}");  time.sleep(2*k) if k<n else None
    print("  WARNING: FRED unreachable — flat static fallback for all dates.")
    return pd.DataFrame([FRED_FALLBACK]*len(PANEL_DATES),index=PANEL_DATES,columns=MAT_YRS)
rf_hist=_fred()
def rf_interp_asof(date):
    row=rf_hist.iloc[rf_hist.index.get_indexer([date],method="nearest")[0]].dropna()
    return interp1d(row.index.astype(float),row.values/100.0,kind="linear",fill_value="extrapolate")

---
## Section 1 — Precompute per-(day, bond) market yield, duration & transaction cost

Heavy step (effective-interest IRR for every priced bond on every backtest day). Set `STEP` to
sub-sample days if you want a faster trial run; `STEP=1` is fully daily.

In [ ]:
# Backtest dates (STEP=1 → every trading day)
STEP = 1
BT_DATES = PANEL_DATES[::STEP]
if BT_DATES[-1] != PANEL_DATES[-1]:
    BT_DATES = BT_DATES.append(PANEL_DATES[-1:])
nD = len(BT_DATES)
mid_A = mid_w.reindex(BT_DATES).values
bid_A = bid_w.reindex(BT_DATES).values
ask_A = ask_w.reindex(BT_DATES).values
print(f"Backtest days: {nD}  (STEP={STEP})")

In [ ]:
# Market book yield Y[d,i] (IRR of remaining CFs vs mid/100) and modified duration DUR[d,i]
Y   = np.full((nD, N), np.nan)
DUR = np.full((nD, N), np.nan)
date64 = BT_DATES.values.astype("datetime64[ns]")
ONE_DAY = np.timedelta64(1, "D")

t0 = time.time()
for i, c in enumerate(CUSIPS):
    if c not in CF_BY:
        continue
    pdates, pcf = CF_BY[c]
    mat = maturity64[i]
    for d in range(nD):
        P = mid_A[d, i]
        if not (np.isfinite(P) and P > 0 and mat > date64[d]):
            continue
        fut = pdates > date64[d]
        if not fut.any():
            continue
        t  = (pdates[fut] - date64[d]) / ONE_DAY / 365.25
        cf = pcf[fut]
        P1 = P / 100.0
        try:
            y = brentq(lambda r: float((cf*(1.0+r)**(-t)).sum() - P1), -0.5, 1.0, maxiter=100)
        except (ValueError, OverflowError):
            continue
        Y[d, i] = y
        disc = (1.0+y)**(-t); pv = cf*disc; tot = pv.sum()
        if tot > 0:
            DUR[d, i] = (t*pv).sum()/tot/(1.0+y)
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/{N} bonds  ({time.time()-t0:.0f}s)")
print(f"IRR/duration precompute done in {time.time()-t0:.0f}s")

In [ ]:
# Transaction cost TAU[d,i] = (ask-bid)/(2*mid); median-filled per day. Eligibility ELIG[d,i].
with np.errstate(invalid="ignore", divide="ignore"):
    TAU = (ask_A - bid_A) / (2.0*mid_A)
TAU = np.where(np.isfinite(TAU) & (TAU > 0), TAU, np.nan)
for d in range(nD):
    row = TAU[d]; fin = np.isfinite(row)
    TAU[d] = np.where(fin, row, np.nanmedian(row[fin]) if fin.any() else 0.0010)

alive = (maturity64[None, :] > date64[:, None])
ELIG  = alive & np.isfinite(mid_A) & (mid_A > 0)
# Coefficients fed to Gurobi must be finite even for non-traded bonds:
Y   = np.nan_to_num(Y,   nan=0.0, posinf=0.0, neginf=0.0)
DUR = np.nan_to_num(DUR, nan=0.0, posinf=0.0, neginf=0.0)
print(f"TAU mean {np.nanmean(TAU)*1e4:.1f} bps | eligibility mean {ELIG.mean():.3f}")

In [ ]:
# Absolute-quarter cash-flow structures (built once; sliced per day for the facility/ALM block)
ALLQ = pd.period_range(cf_all.PaymentDate.min().to_period("Q"), FABN_MATURITY.to_period("Q"), freq="Q")
qpos = {q:k for k,q in enumerate(ALLQ)}
QB = np.zeros((len(ALLQ), N))                       # quarterly bond CF, per $1 face
tmp = cf_all.copy(); tmp["q"] = tmp.PaymentDate.dt.to_period("Q")
for (q,c),g in tmp.groupby(["q","CUSIP"]):
    if q in qpos and c in cidx: QB[qpos[q], cidx[c]] += g.Payment.sum()/100.0

# FABN semi-annual schedule → quarterly liability ($), and remaining-duration helper
_sd = pd.DatetimeIndex([FABN_ISSUE + pd.DateOffset(months=6*k) for k in range(1,11)])
_tot = np.full(len(_sd), FABN_COUPON/2*100.0); _tot[-1] += 100.0
FABN_SCHED = pd.DataFrame({"date":_sd, "total":_tot})
FB = np.zeros(len(ALLQ))
for _,r in FABN_SCHED.iterrows():
    q = r["date"].to_period("Q")
    if q in qpos: FB[qpos[q]] += r["total"]*(H/100.0)
# Bonds with CFs in quarters after the FABN's last paying quarter cannot fund it
_fabn_last = max(k for k in range(len(ALLQ)) if FB[k] > 0)
POST_FABN = np.any(QB[_fabn_last+1:] > 1e-6, axis=0) if _fabn_last+1 < len(ALLQ) else np.zeros(N,bool)
print(f"Quarter grid {ALLQ[0]}..{ALLQ[-1]} | post-FABN bonds excluded: {int(POST_FABN.sum())}")

def quarter_slice(date):
    """Remaining-quarter bond CF (Q×N), FABN liability (Q,), discount factors, PV(L), D_FABN."""
    q0 = date.to_period("Q"); k0 = qpos[q0]
    qb = QB[k0:]; fb = FB[k0:]; Q = len(fb); dt_q = 0.25
    df = np.array([(1.0+r_FABN)**(-(dt_q*(k+1))) for k in range(Q)])
    fut = FABN_SCHED[FABN_SCHED.date > date]
    if len(fut):
        tt = (fut.date - date).dt.days/365.25; pv = fut.total.sum()
        D_FABN = (tt*fut.total).sum()/pv/(1.0+r_FABN/2)
    else:
        D_FABN = 0.0
    return qb, fb, df, float((fb*df).sum()), D_FABN, dt_q

print("quarter_slice() ready.")

---
## Section 2 — Per-Day SAP Solver (buy/sell decomposition, all four cost terms)

In [ ]:
def solve_day(d, h_prev, y_book, P):
    """One daily SAP solve. Returns (h, b, s, status). d = index into BT_DATES."""
    date = BT_DATES[d]
    ymkt = Y[d]; durs = DUR[d]; tau = TAU[d]; elig = ELIG[d]
    qb, fb, df, PV_L, D_FABN, dt_q = quarter_slice(date)
    Q = len(fb)
    T = max((FABN_MATURITY - date).days/365.25, 1e-6)        # decision horizon (yrs to maturity)
    lam = P["cost_of_capital"]*P["RBC_bar"]

    m = gp.Model("sap_day"); m.Params.LogToConsole=0; m.Params.OutputFlag=0
    b = m.addVars(N, lb=0.0)                                  # buys
    s = m.addVars(N, lb=0.0)                                  # sells
    for i in range(N):
        s[i].ub = float(h_prev[i])                           # cannot sell more than held
        if (not elig[i]) or POST_FABN[i]:                    # cannot buy ineligible / post-FABN bonds
            b[i].ub = 0.0
    h = [h_prev[i] + b[i] - s[i] for i in range(N)]          # linear expressions
    dp = m.addVar(lb=0.0); dn = m.addVar(lb=0.0)
    B  = m.addVars(Q, lb=0.0); sn = m.addVars(Q, lb=0.0)

    # Income: retained at locked book yield, new buys at market yield (net of funding)
    nii = gp.quicksum((h_prev[i]-s[i])*(y_book[i]-r_FABN) + b[i]*(ymkt[i]-r_FABN) for i in range(N))
    cap = gp.quicksum(theta[i]*h[i] for i in range(N))
    trd = gp.quicksum(tau[i]*(b[i]+s[i]) for i in range(N))
    lend   = P["r_save"]   * dt_q * gp.quicksum(B[q]  for q in range(Q))
    borrow = P["r_borrow"] * dt_q * gp.quicksum(sn[q] for q in range(Q))
    m.setObjective(T*nii - T*lam*cap - trd + lend - borrow, GRB.MAXIMIZE)

    m.addConstr(gp.quicksum(h[i] for i in range(N)) == H)
    m.addConstr(gp.quicksum(durs[i]*h[i] for i in range(N)) - D_FABN*H == dp - dn)
    m.addConstr(dp <= P["eps_D"]*H); m.addConstr(dn <= P["eps_D"]*H)
    for g,idxs in ISSUER_GROUPS.items():
        m.addConstr(gp.quicksum(h[i] for i in idxs) <= P["delta_iss"]*H)
    for q in range(Q):
        cfa = gp.quicksum(qb[q,i]*h[i] for i in range(N)); cfl = float(fb[q])
        if q==0: m.addConstr(B[q]-sn[q] == cfa - cfl)
        else:    m.addConstr(B[q]-sn[q] == (1.0+P["r_save"]*dt_q)*B[q-1] + cfa - cfl)
    m.addConstr(gp.quicksum(df[q]*sn[q] for q in range(Q)) <= P["phi_sf"]*PV_L)

    m.optimize()
    if m.Status == GRB.OPTIMAL:
        bb = np.array([b[i].X for i in range(N)]); ss = np.array([s[i].X for i in range(N)])
        hh = h_prev + bb - ss
        return np.maximum(hh,0.0), bb, ss, "OPTIMAL"
    return h_prev.copy(), np.zeros(N), np.zeros(N), f"STATUS_{m.Status}_hold"

print("solve_day() ready.")

---
## Section 3 — Daily Backtest (Dynamic vs Static)

In [ ]:
PARAMS = dict(
    cost_of_capital=0.08,  # WACC on required capital (annual)
    RBC_bar        =1.5,   # required-capital multiplier on C-1
    eps_D          =0.30,  # duration band (yrs)
    delta_iss      =0.05,  # issuer concentration cap (5% of H)
    r_save         =r_FABN,# lending/reinvestment rate on facility surplus
    r_borrow       =0.05,  # borrowing rate on facility shortfall
    phi_sf         =0.01,  # PV shortfall cap (fraction of PV liability)
)
LAMBDA = PARAMS["cost_of_capital"]*PARAMS["RBC_bar"]
print("lambda (capital charge) =", LAMBDA)

In [ ]:
def run_daily(dynamic=True):
    """Daily backtest with amortized-cost in-force-yield ledger.
       Dynamic: re-optimize every day. Static: optimize day 0, then hold (matured→cash @ rF)."""
    h_prev = np.zeros(N); y_book = np.zeros(N); recs = []
    h0 = None
    for d in range(nD):
        date = BT_DATES[d]
        # Day length (years) to next backtest date
        dt = (BT_DATES[d+1]-date).days/365.25 if d+1 < nD else (PANEL_DATES.max()-date).days/365.25
        dt = max(dt, 0.0)

        # ── Redeem matured bonds at par (no spread); cash returns to budget ───
        matured = (~ELIG[d]) & (maturity64 <= date.to_datetime64()) & (h_prev > 0)
        if matured.any():
            h_prev = h_prev.copy(); h_prev[matured] = 0.0; y_book = y_book.copy(); y_book[matured] = 0.0

        if dynamic or d == 0:
            h_k, bb, ss, status = solve_day(d, h_prev, y_book, PARAMS)
        else:
            # static hold: no trading; freeze book (matured already redeemed above)
            h_k, bb, ss, status = h_prev.copy(), np.zeros(N), np.zeros(N), "HOLD"

        # ── Book-yield ledger update (retained keep locked yield; buys at market) ─
        retained = h_prev - ss
        with np.errstate(invalid="ignore", divide="ignore"):
            y_book = np.where(h_k > 1e-9,
                              (retained*y_book + bb*Y[d])/np.where(h_k>1e-9, h_k, 1.0), 0.0)

        # ── Realized daily P&L (accrual at in-force book yield) ──────────────
        # Accrue on ALIVE bonds (a held bond earns regardless of a quote that day),
        # not only priced ones; matured positions were already redeemed to cash above.
        e = alive[d].astype(float)
        nii = dt * float(np.sum(e*h_k*(y_book - r_FABN)))
        cap = dt * LAMBDA * float(np.sum(e*theta*h_k))
        trd = float(np.sum(TAU[d]*(bb+ss)))
        net = nii - cap - trd
        sel = h_k > 1.0
        recs.append(dict(d=d, date=date, dt=dt, status=status, NII=nii, Capital=cap,
                         Turnover=trd, Net=net, CumNet=net+(recs[-1]["CumNet"] if recs else 0.0),
                         wavg_book_yield=float(np.sum(y_book[sel]*h_k[sel])/h_k[sel].sum()) if sel.any() else np.nan,
                         req_capital=PARAMS["RBC_bar"]*float(np.sum(e*theta*h_k)),
                         n_held=int(sel.sum()), n_trades=int(np.sum((bb+ss) > 1.0)),
                         traded_notional=float(np.sum(bb+ss))))
        if dynamic and (d+1) % 50 == 0:
            print(f"  ...day {d+1}/{nD}  {date.date()}  cumNet=${recs[-1]['CumNet']/1e6:.2f}M")
        h_prev = h_k
        if d == 0: h0 = h_k.copy()
    return pd.DataFrame(recs), h0

print("Running DYNAMIC daily backtest (this is the slow one)...")
dyn_df, h0 = run_daily(dynamic=True)
print("Running STATIC daily backtest (hold day-0 book)...")
sta_df, _  = run_daily(dynamic=False)
print("Done.")

---
## Section 4 — Results

In [ ]:
dyn_total, sta_total = dyn_df["Net"].sum(), sta_df["Net"].sum()
print("="*66)
print(f"  DYNAMIC cumulative net statutory income : ${dyn_total:,.0f}")
print(f"  STATIC  cumulative net statutory income : ${sta_total:,.0f}")
adv = dyn_total - sta_total
print(f"  Dynamic advantage                       : ${adv:,.0f}  ({adv/abs(sta_total)*100:+.2f}%)")
print(f"  Dynamic total turnover paid             : ${dyn_df['Turnover'].sum():,.0f}")
print(f"  Total trade-days (dynamic)              : {int((dyn_df['n_trades']>0).sum())}/{nD}")
print(f"  Total notional traded (dynamic)         : ${dyn_df['traded_notional'].sum():,.0f}")
print("="*66)
display(dyn_df[["date","status","NII","Capital","Turnover","Net","CumNet","n_trades","wavg_book_yield"]].head(12))

In [ ]:
# Fig 1: cumulative net statutory income, dynamic vs static
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(dyn_df["date"], dyn_df["CumNet"], color="#2980b9", lw=1.8, label="Dynamic (daily re-opt)")
ax.plot(sta_df["date"], sta_df["CumNet"], color="#e67e22", lw=1.8, ls="--", label="Static buy-and-hold")
ax.set_title("Cumulative Net Statutory Income — Daily Dynamic vs Static")
ax.set_ylabel("Cumulative net ($)"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f"${v/1e6:.1f}M"))
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Fig 2: in-force weighted book yield over time
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(dyn_df["date"], dyn_df["wavg_book_yield"]*100, color="#2980b9", lw=1.5, label="Dynamic in-force yield")
ax.plot(sta_df["date"], sta_df["wavg_book_yield"]*100, color="#e67e22", lw=1.5, ls="--", label="Static (locked at day 0)")
ax.axhline(r_FABN*100, color="grey", ls=":", label=f"r_FABN = {r_FABN*100:.2f}%")
ax.set_title("Weighted-Average In-Force Book Yield"); ax.set_ylabel("Yield (%)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Fig 3: daily turnover paid + trade count (dynamic)
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(dyn_df["date"], dyn_df["Turnover"], width=2.0, color="#c0392b", label="Daily turnover cost")
ax.set_ylabel("Turnover cost ($)"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f"${v/1e3:.0f}k"))
ax2 = ax.twinx(); ax2.plot(dyn_df["date"], dyn_df["traded_notional"]/1e6, color="#7f8c8d", lw=1, label="Notional traded ($M)")
ax2.set_ylabel("Notional traded ($M)")
ax.set_title("Dynamic Trading Activity (when opportunities clear the cost hurdle)")
ax.legend(loc="upper left"); ax2.legend(loc="upper right"); plt.tight_layout(); plt.show()

# Fig 4: required capital over time
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(dyn_df["date"], dyn_df["req_capital"], color="#2980b9", lw=1.5, label="Dynamic")
ax.plot(sta_df["date"], sta_df["req_capital"], color="#e67e22", lw=1.5, ls="--", label="Static")
ax.set_title("Required Capital  (RBC_bar x Sum theta_i h_i)"); ax.set_ylabel("Required capital ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f"${v/1e6:.1f}M"))
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## Notes

- **Daily decision, horizon-valued:** each day weighs a holding over its remaining life to FABN
  maturity, so a one-time bid-ask cost is judged against the income it earns over the horizon — that
  is why worthwhile opportunities are taken instead of churning or freezing.
- **All four economics in the swap:** trading cost (real bid-ask, both buys and sells), capital-cost
  change ($T\lambda\theta h$), lending revenue ($r_{save}$ on facility surplus), borrowing cost
  ($r_{borrow}$ on shortfall).
- **Amortized cost:** retained lots keep their locked yield; only new buys re-strike at market — the
  true "pickup minus what you give up."
- **Runtime:** the IRR precompute (Section 1) is the slow step; set `STEP>1` for a quick trial.
- **Conservative test:** IMR/AVR excluded, so no trading gains are booked — the dynamic edge here is
  purely forward book-yield pickup net of costs and reinvestment of run-off, not realized P&L on sales.